In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import torch
import warnings
warnings.filterwarnings("ignore")

# 导入 TabNet
from pytorch_tabnet.tab_model import TabNetRegressor

# 1. 加载数据
print("正在加载数据...")
# 使用与LightGBM相同的平滑数据集
df_original = pd.read_excel('../../data/quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量 - 与LightGBM保持一致
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
_search_cut = int(len(X_train) * 0.8)
scaler_X = StandardScaler().fit(X_train.iloc[:_search_cut])
scaler_y = StandardScaler().fit(y_train.iloc[:_search_cut])
X_train_scaled = scaler_X.transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.transform(y_train)
X_search_train_scaled = X_train_scaled[:_search_cut]
X_search_val_scaled = X_train_scaled[_search_cut:]
y_search_train_scaled = y_train_scaled[:_search_cut]
y_search_val_scaled = y_train_scaled[_search_cut:]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
# 最优参数:
# n_d: 22
# n_a: 128
# n_steps: 3
# gamma: 2.0
# lambda_sparse: 0.0005467072969055289
# lr: 0.029384293039604333
# batch_size: 115
# virtual_batch_size: 119
# 7. 创建TabNet模型
print("\n开始训练TabNet模型...")
tabnet_model = TabNetRegressor(
    n_d=22,                      # 决策层维度
    n_a=128,                     # 注意力层维度
    n_steps=3,                   # 步数
    gamma=2.0,                   # 稀疏度
    lambda_sparse=0.0005467072969055289,  # 额外的稀疏度
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.029384293039604333),  # 学习率
    scheduler_params={"step_size":50, "gamma":0.8},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    device_name=device,
    verbose=0,   # 显示训练进度
    seed=42
)

# 8. 训练TabNet模型
tabnet_model.fit(
    X_train=X_search_train_scaled,
    y_train=y_search_train_scaled,
    eval_set=[(X_search_val_scaled, y_search_val_scaled)],
    eval_name=['val'], eval_metric=['rmse'],
    max_epochs=200, patience=20, batch_size=227,
    virtual_batch_size=96, num_workers=0
)
selected_epochs = max(1, int(getattr(tabnet_model, 'best_epoch', 100)))

# Fresh final fit on all outer-training records; the holdout is not used for stopping.
tabnet_model = TabNetRegressor(
    n_d=22, n_a=128, n_steps=3, gamma=2.0,
    lambda_sparse=0.0005467072969055289,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.029384293039604333),
    scheduler_params={"step_size":50, "gamma":0.8},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax', device_name=device, verbose=0, seed=42
)
tabnet_model.fit(
    X_train=X_train_scaled, y_train=y_train_scaled,
    max_epochs=selected_epochs, patience=0, batch_size=227,
    virtual_batch_size=96, num_workers=0
)
print("TabNet model refit completed using training rows only.")

y_pred_scaled = tabnet_model.predict(X_test_scaled)

# 10. 将预测结果反标准化回原始空间
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# 11. 评估模型性能
print("\n模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")